In [1]:
from pprint import pprint
import pandas as pd
from omegaconf import OmegaConf
from embedding_train.config import load_base_config
from embedding_train.data import EmbeddingDataModule
# load the same base config training uses
cfg = load_base_config()
# optional overrides for notebook inspection
cfg = OmegaConf.merge(
    cfg,
    {
        "data": {
            "num_workers": 0,      # safer in notebooks
            "pin_memory": False,
            "limit_rows": 256,     # keep setup fast; remove for full dataset
            # "train_batching_mode": "anchor_query",   # or "random_pairs"
        }
    },
)
dm = EmbeddingDataModule(cfg)
dm.setup()
print(dm.dataset_stats)
print("train rows:", len(dm.train_dataset))
print("val rows:", len(dm.val_dataset))
# inspect raw rendered training records before tokenization
pd.DataFrame(dm.train_dataset.records).head(10)
# inspect one batch exactly as the model sees it after collate/tokenization
batch = next(iter(dm.train_dataloader()))
preview = pd.DataFrame({
    "query_id": batch["query_ids"],
    "offer_id": batch["offer_ids"],
    "label": batch["labels"].tolist(),
    "raw_label": batch["raw_labels"],
    "query_text": [
        dm.tokenizer.decode(ids, skip_special_tokens=True)
        for ids in batch["query_inputs"]["input_ids"]
    ],
    "offer_text": [
        dm.tokenizer.decode(ids, skip_special_tokens=True)
        for ids in batch["offer_inputs"]["input_ids"]
    ],
    "query_tokens": [
        int(mask.sum()) for mask in batch["query_inputs"]["attention_mask"]
    ],
    "offer_tokens": [
        int(mask.sum()) for mask in batch["offer_inputs"]["attention_mask"]
    ],
})
display(preview)
if "batch_stats" in batch:
    pprint(batch["batch_stats"])

/Users/max/Clients/simplesystem/code/embedding/main/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/max/Clients/simplesystem/code/embedding/main/.venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Loaded dataset: {'train_rows': 243, 'val_rows': 13, 'train_queries': 235, 'val_queries': 12, 'train_offers': 243, 'val_offers': 13, 'train_positive_rows': 175, 'val_positive_rows': 8, 'train_negative_rows': 68, 'val_negative_rows': 5, 'val_query_share': 0.048582995951417005, 'val_offer_share': 0.05078125, 'val_row_share': 0.05078125, 'train_positive_rate': 0.720164609053498, 'val_positive_rate': 0.6153846153846154, 'shared_offers_between_train_and_val': 0, 'skipped_rows': 0, 'eligible_train_queries': 6, 'target_val_queries': 12, 'selected_val_queries': 12, 'connected_components': 247, 'train_connected_components': 235, 'val_connected_components': 12}
{'train_rows': 243, 'val_rows': 13, 'train_queries': 235, 'val_queries': 12, 'train_offers': 243, 'val_offers': 13, 'train_positive_rows': 175, 'val_positive_rows': 8, 'train_negative_rows': 68, 'val_negative_rows': 5, 'val_query_share': 0.048582995951417005, 'val_offer_share': 0.05078125, 'val_row_share': 0.05078125, 'train_positive_rate'

,query_id,offer_id,label,raw_label,query_text,offer_text,query_tokens,offer_tokens
0,34fc60e0-488b-5983-8fc0-bdcb243efbe3,ma9J0FGzQ3+g2sf+VLXnkg==,1.0,Exact,Innensechskantschrauben,Article Name: RS PRO Zylinderkopf Innensechska...,6,256
1,2a80a0b8-a145-5b7b-8617-ce05c2a0643a,BuP9s1+CRq+4CkcE9Rqtvg==,1.0,Exact,Edding 3000,Article Name: edding Permanentmarker 3000 4-30...,3,102
2,26b7e93b-bb73-5c9e-87f6-b86cce930cea,YsuINalASWGWilu43k0KLA==,0.0,Substitute,KÜhlschrank,Article Name: C+P Kühlschrankcaddy L+E- Tassen...,4,250
3,d7385c55-7eda-5753-a86c-9902a114fd9d,e3nnrlH+QgSFJslLOVrLEg==,1.0,Exact,6311-2Z,Article Name: Rillenkugellager mit Dichtungen ...,3,195
4,dfb7331a-6a69-56da-9b07-adf9af90e2e4,u5uK+ZGVRyKB/ubc0G7gpA==,1.0,Exact,3012,"Article Name: TDK VLS-HBX Drosselspule Ja, 1 μ...",2,117
5,e51b0c54-81f3-5a7c-a5e5-e07fd4ecc200,/W+aovXFQK+nbcHBHG8s3g==,1.0,Exact,Batterie Baby,"Article Name: TADIRAN Batterie Lithium 3,6V Ba...",2,188
6,e71d7328-0056-5504-86ac-a4b77da05f17,fgR1CMNQQiaMfrHa/v+vPg==,1.0,Exact,EInsatzkasten,"Article Name: Einsatzkasten EK 4041, blau EAN:...",4,164
7,1732f542-4732-5bd1-a419-6890c491aa57,KCEkm8jYR9mAXCgbHNfi7Q==,1.0,Exact,Holzstiel,Article Name: Kärcher Holzstiel D28 EAN: 00000...,3,104
8,48ab5554-7f0c-5793-8199-ad5f7a07164e,J+M3Rr1fRqCUwtx4j9KF9g==,1.0,Exact,Laminierfolie A3,Article Name: Fellowes Laminierfolie ImageLast...,5,141
9,2b38e1bd-acd4-534c-a56e-c0ccdf6f1c3c,eG+ijiDXQNOz2N96Ly312Q==,1.0,Exact,6204 zz,Article Name: Rillenkugellager 6204 2Z.C3 EAN:...,4,78
